# Arrhythmia — ablação da atenção entre amostras + painel (DUAS T4, Kaggle)

Regime em que o artigo do SAINT relata ganho da atenção entre amostras (poucas amostras, muitos atributos: 452 × 279).
Script: `scripts/study_arrhythmia_kaggle.py`. 9 modelos × 10 sementes, 200 épocas sem parada antecipada (melhor checkpoint de validação):

| modelo | o que varia |
|---|---|
| SAINT com / sem MISA | só a MISA (config. do artigo p/ Arrhythmia: d=8, h=4, lr=1e-4, lote 32) |
| FT-CUR CUR / sem inter / densa | só a atenção entre amostras |
| FT-Softmax / FT-Sparsemax | só a função de atenção entre atributos |
| ADMM-Nyström (ℓ1), XGBoost | referências |

**Antes de rodar:** Settings → Accelerator → **GPU T4 x2**; Internet **ligada** (clone + dados da UCI).
A célula 3 é um **teste rápido** (1 semente, 2 épocas): se algum modelo der erro, ela para antes da execução completa.
**Não cancele a sessão** — o Kaggle só garante a saída quando ela termina sozinha. Estimativa: ≈1–2 h.

In [ ]:
# ── 1. GPUs, repositório e dados ──
import os, subprocess, sys, urllib.request
from pathlib import Path
import torch
print('CUDA:', torch.cuda.is_available(), '| placas:', torch.cuda.device_count())
assert torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2'
GIT_URL='https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR='/kaggle/working/sparse-lssvm-transformers-study'; BRANCH='revisao/estatistica-e-proveniencia'
if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git','clone','--branch',BRANCH,GIT_URL,PROJECT_DIR],check=True)
subprocess.run(['git','-C',PROJECT_DIR,'pull','--rebase','origin',BRANCH],check=True)
os.chdir(PROJECT_DIR); sys.path.insert(0,'.')
!git log --oneline -1
assert Path('scripts/study_arrhythmia_kaggle.py').exists(), 'script ausente — faça o push e refaça o pull'
!pip install -q einops 2>&1 | tail -n 1
d = Path('data/raw/arrhythmia'); d.mkdir(parents=True, exist_ok=True)
if not (d/'arrhythmia.data').exists():   # data/raw/ não é versionado
    urllib.request.urlretrieve('https://archive.ics.uci.edu/ml/machine-learning-databases/arrhythmia/arrhythmia.data', d/'arrhythmia.data')
import numpy as np
raw = np.genfromtxt(d/'arrhythmia.data', delimiter=',', missing_values='?', filling_values=np.nan)
assert raw.shape == (452, 280), raw.shape
print('dados OK', raw.shape)

In [ ]:
# ── 2. Execução em duas placas (metade das sementes cada; um JSON por placa) ──
import json, shutil, time
OUT = 'results/study_arrhythmia_kaggle.json'
def run_2gpu(extra, seeds, out, every=60):
    half = len(seeds)//2; procs, shards, logs = [], [], []
    for gpu, sds in [(0, seeds[:half]), (1, seeds[half:])]:
        shard = out.replace('.json', f'_g{gpu}.json'); log = f'/kaggle/working/{Path(shard).stem}.log'
        shards.append(shard); logs.append(log)
        cmd = f"python -u scripts/study_arrhythmia_kaggle.py --seeds {' '.join(map(str, sds))} --output {shard} {extra}"
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
        procs.append(subprocess.Popen(cmd, shell=True, stdout=open(log,'w'), stderr=subprocess.STDOUT, env=env))
    t0 = time.time(); last = -1
    while True:
        rcs = [p.poll() for p in procs]; recs = []
        for s in shards:
            try: recs += json.load(open(s))
            except (FileNotFoundError, ValueError): pass
        if len(recs) != last or all(r is not None for r in rcs):
            print(f'{len(recs):>3} registros  {(time.time()-t0)/60:5.0f} min', flush=True); last = len(recs)
            for s in shards:
                if Path(s).exists(): shutil.copy(s, '/kaggle/working/' + Path(s).name)
        if all(r is not None for r in rcs): break
        time.sleep(every)
    for lg in logs: print('---', Path(lg).name); print(''.join(open(lg).readlines()[-12:]))
    Path(out).write_text(json.dumps(recs, indent=1)); shutil.copy(out, '/kaggle/working/' + Path(out).name)
    return recs, rcs
print('ok')

In [ ]:
# ── 3. TESTE RÁPIDO: 1 semente por placa, 2 épocas, os 9 modelos ──
recs, rcs = run_2gpu('--smoke', [0, 1], 'results/arr_smoke.json', every=15)
bad = [(r['model'], r.get('error')) for r in recs if r['status'] != 'ok']
for r in recs: print(f"  {r['seed']} {r['model']:22s} {r['status']:5s} {r.get('peak_gpu_mib', float('nan')):7.0f} MiB  {r['fit_s']:5.0f}s")
assert rcs == [0, 0] and not bad and len(recs) == 18, f'teste rápido falhou: {bad or rcs}'
print('TESTE RÁPIDO OK — seguindo para a execução completa')

In [ ]:
# ── 4. EXECUÇÃO COMPLETA: 10 sementes (0–4 na GPU 0, 5–9 na GPU 1) ──
recs, rcs = run_2gpu('', list(range(10)), OUT)
print('exits', rcs)

In [ ]:
# ── 5. Resumo ──
import collections, statistics as st
by = collections.defaultdict(list)
for r in recs:
    if r['status'] == 'ok': by[r['model']].append((r['test_f1_macro'], r['test_auc']))
for m, v in by.items():
    print(f"{m:22s} n={len(v):2d}  F1 {st.mean(a for a,_ in v):.3f}  AUC {st.mean(b for _,b in v):.3f}")
err = [(r['seed'], r['model'], r.get('error')) for r in recs if r['status'] != 'ok']
print('erros:', err or 'nenhum')